In [4]:
import pandas as pd
import numpy as np
from itertools import combinations

In [5]:
# Load CSVs into DataFrames
users_df = pd.read_csv("users.csv")
groups_df = pd.read_csv("groups.csv")
memberships_df = pd.read_csv("group_memberships.csv")

In [ ]:
# Import all matching helper functions from scorehelper modules
from scorehelper.locationMatch import location_match  # Checks if locations match (dealbreaker)
from scorehelper.groupSizeMatch import size_compat   # Computes group size compatibility
from scorehelper.groupAgeMatch import age_overlap_score  # Computes age range overlap
from scorehelper.lifeStyleMatch import lifestyle_similarity  # Computes lifestyle similarity
from scorehelper.languageMatch import jaccard_array  # Computes language similarity (Jaccard index)
from scorehelper.badgeMatch import inclusivity_score, accessibility_score  # Inclusivity/accessibility scores
from scorehelper.badgeMatch import rating_score  # Computes average group rating score
from itertools import combinations  # Used to generate all unique group pairs

# Define the weights for each matching component (how much each factor matters)
DEFAULT_WEIGHTS = {
    "size": 0.20,                # Weight for group size compatibility
    "age": 0.15,                 # Weight for age range overlap
    "lifestyle": 0.20,           # Weight for lifestyle similarity
    "languages": 0.15,           # Weight for language similarity
    "inclusivity_access": 0.10,  # Weight for inclusivity/accessibility
    "rating": 0.10,              # Weight for group rating
    "location": 0.10,            # Weight for location match (dealbreaker)
}

# Main function to compute match scores for all group pairs
def compute_pair_scores(groups_df: pd.DataFrame,
                        group_languages_df: pd.DataFrame,
                        languages_df: pd.DataFrame,
                        weights: dict = None,
                        same_city_only: bool = False) -> pd.DataFrame:
    """
    Returns a DataFrame with all unordered pairs and a 0–100 match_score,
    plus the sub-scores for explainability.
    """
    w = (weights or DEFAULT_WEIGHTS).copy()  # Use provided weights or defaults

    required = [
        "id","age_range","num_men","num_women","num_nonbinary","location_id",
        "smoking_level","drinking_level","weed_level","ideal_group_size",
        "sexuality_inclusive","accessibility_friendly","group_rating"
    ]
    missing = [c for c in required if c not in groups_df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Build a mapping from group_id to set of language names
    lang_id_to_name = dict(zip(languages_df["id"], languages_df["name"]))
    group_to_langs = group_languages_df.groupby("group_id")["language_id"].apply(
        lambda ids: set(lang_id_to_name[i] for i in ids if i in lang_id_to_name)
    ).to_dict()

    rows = []
    by_id = {int(r["id"]): r for _, r in groups_df.iterrows()}

    for a_id, b_id in combinations(sorted(by_id.keys()), 2):
        a, b = by_id[a_id], by_id[b_id]

        # Location dealbreaker: if locations differ, skip this pair
        loc_score = location_match(a["location_id"], b["location_id"])
        if loc_score is None:
            continue  # Dealbreaker: skip this pair

        # Get language sets for each group
        langs_a = group_to_langs.get(a_id, set())
        langs_b = group_to_langs.get(b_id, set())
        lang_score = jaccard_array(langs_a, langs_b)  # Language similarity

        size_score = size_compat(a, b)  # Group size compatibility
        age_score = age_overlap_score(a, b)  # Age range overlap
        life_score = lifestyle_similarity(a, b)  # Lifestyle similarity
        inc = inclusivity_score(a, b)  # Sexuality inclusivity
        acc = accessibility_score(a, b)  # Accessibility friendliness
        inc_acc = (inc + acc) / 2.0  # Average of inclusivity and accessibility
        rate = rating_score(a, b)  # Average group rating
        city = loc_score  # Location score (always 1 if matched, else skipped)

        match_score = (
            w["size"] * size_score +
            w["age"] * age_score +
            w["lifestyle"] * life_score +
            w["languages"] * lang_score +
            w["inclusivity_access"] * inc_acc +
            w["rating"] * rate +
            w["location"] * city
        ) * 100.0

        rows.append({
            "a_id": a_id, "b_id": b_id,
            "match_score": round(match_score, 1),
            "size_score": round(size_score, 3),
            "age_score": round(age_score, 3),
            "lifestyle_score": round(life_score, 3),
            "lang_score": round(lang_score, 3),
            "inclusivity_access_score": round(inc_acc, 3),
            "rating_score": round(rate, 3),
            "city_score": round(city, 3),
        })

    return pd.DataFrame(rows).sort_values(["match_score","a_id","b_id"], ascending=[False, True, True]).reset_index(drop=True)



All pair scores:
   a_id  b_id  match_score  size_score  age_score  lifestyle_score  \
0     2     4         84.7         0.7      0.714            0.852   
1     1     3         76.4         0.8      0.667            0.852   
2     1     2         70.1         0.9      0.375            0.889   
3     2     3         69.9         0.9      0.571            0.741   
4     3     4         62.1         0.8      0.375            0.593   
5     1     4         58.9         0.6      0.222            0.741   

   lang_score  inclusivity_access_score  rating_score  city_score  
0       1.000                       1.0          0.79         1.0  
1       0.333                       1.0          0.84         1.0  
2       0.500                       1.0          0.82         0.3  
3       0.500                       1.0          0.80         0.3  
4       0.500                       1.0          0.81         0.3  
5       0.500                       1.0          0.83         0.3  

Best match per 